In [ ]:
import dataclasses

import numpy as np
import pandas as pd

from gbp.build.pipeline import build_model
from gbp.consumers.simulator.built_in_phases import (
    ArrivalsPhase,
    DeparturePhysicsPhase,
    HistoricalLatentDemandPhase,
    HistoricalODStructurePhase,
    HistoricalTripSamplingPhase,
    InvariantCheckPhase,
    OverflowRedirectPhase,
)
from gbp.consumers.simulator.config import EnvironmentConfig
from gbp.consumers.simulator.engine import Environment
from gbp.consumers.simulator.state import init_state
from gbp.loaders.contracts import GraphLoaderConfig
from gbp.loaders.dataloader_graph import DataLoaderGraph
from gbp.loaders.dataloader_mock import DataLoaderMock
from gbp.consumers.simulator.dispatch_phase import DispatchPhase
from gbp.consumers.simulator.phases import Schedule
from gbp.consumers.simulator.tasks.rebalancer import RebalancerTask

mock_config = {
    "n_stations": 10,
    "n_depots": 2,
    "n_timestamps": 72,
    "time_freq": "h",
    "start_date": "2025-01-01",
    "ebike_fraction": 0.3,
    "depot_capacity": 200,
    "seed": 42,
}
mock_source = DataLoaderMock(mock_config, n_trucks=3, truck_capacity_bikes=20)
graph_loader = DataLoaderGraph(mock_source, GraphLoaderConfig())
raw_model = graph_loader.load()
resolved_with_supply = build_model(raw_model)
resolved_model = dataclasses.replace(resolved_with_supply, supply=None)

state_initial = init_state(resolved_model)
inventory_totals_initial = (
    state_initial.inventory.groupby("commodity_category")["quantity"].sum()
)
in_transit_totals_initial = (
    state_initial.in_transit.groupby("commodity_category")["quantity"].sum()
    if not state_initial.in_transit.empty
    else pd.Series(dtype=float)
)
baseline_per_commodity = (
    inventory_totals_initial.add(in_transit_totals_initial, fill_value=0.0)
    .to_dict()
)

rebalance_every_n_periods = 6
rebalancer_commodity = "electric_bike"
rebalancer_task = RebalancerTask(
    min_threshold=0.3,
    max_threshold=0.7,
    time_limit_seconds=5,
    commodity_category=rebalancer_commodity,
)

phases_canonical = [
    HistoricalLatentDemandPhase(),
    HistoricalODStructurePhase(),
    DeparturePhysicsPhase(mode="permissive"),
    HistoricalTripSamplingPhase(use_durations=True),
    ArrivalsPhase(),
    OverflowRedirectPhase(),
    DispatchPhase(
        rebalancer_task,
        schedule=Schedule.every_n(rebalance_every_n_periods),
    ),
    InvariantCheckPhase(
        baseline=baseline_per_commodity,
        fail_on_violation=False,
    ),
]
env_canonical = Environment(
    resolved_model,
    EnvironmentConfig(
        phases=phases_canonical, seed=42, scenario_id="historical_replay",
    ),
)
env_canonical.run()
log_canonical = env_canonical.log.to_dataframes()